# Source audio analysis
This notebook reads the preserved Hugging Face source snapshot under `sources/`.
For model training, use the organized `train.csv` and `test.csv` at the dataset root; see `dataset_splits.ipynb`.

In [ ]:
from pathlib import Path
import subprocess
import wave

import imageio_ffmpeg
import pandas as pd

root = next((parent / "data" for parent in (Path.cwd(), *Path.cwd().parents)
             if (parent / "data/karakalpak-audio-dataset-hf/sources/train.csv").is_file()), None)
if root is None:
    raise FileNotFoundError("karakalpak-audio-dataset-hf/sources/train.csv")
hf_dir = root / "karakalpak-audio-dataset-hf" / "sources"
source_dir = hf_dir / "data"
output_dir = hf_dir / "data_16k"
ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()

def sample_rate(path):
    with wave.open(str(path), "rb") as audio:
        return audio.getframerate()

source_files = sorted(source_dir.glob("*.wav"), key=lambda path: (sample_rate(path), path.name))

In [ ]:
frames = [pd.read_csv(hf_dir / f"{split}.csv") for split in ("train", "test")]
frame = pd.concat(frames, ignore_index=True).set_index("filename", verify_integrity=True)
audio_names = [path.name for path in source_files]
if set(frame.index) != set(audio_names):
    raise ValueError("CSV filenames must match the audio filenames")
frame = frame.loc[audio_names].reset_index()
frame.insert(0, "audio_index", range(len(frame)))
len(frame)

In [ ]:
output_dir.mkdir(exist_ok=True)
for source in source_files:
    destination = output_dir / source.name
    with wave.open(str(source), "rb") as original:
        source_duration = original.getnframes() / original.getframerate()
    valid = False
    if destination.is_file():
        try:
            with wave.open(str(destination), "rb") as converted:
                valid = (
                    (converted.getframerate(), converted.getnchannels(), converted.getsampwidth()) == (16_000, 1, 2)
                    and abs(source_duration - converted.getnframes() / 16_000) < 1 / 16_000
                )
        except (OSError, EOFError, wave.Error):
            pass
    if valid:
        continue
    temporary = destination.with_suffix(".wav.partial")
    subprocess.run(
        [ffmpeg, "-nostdin", "-loglevel", "error", "-threads", "1", "-y",
         "-i", str(source), "-ar", "16000", "-ac", "1",
         "-c:a", "pcm_s16le", "-f", "wav", str(temporary)],
        check=True,
    )
    temporary.replace(destination)

In [ ]:
if {path.name for path in output_dir.glob("*.wav")} != set(audio_names):
    raise ValueError("data_16k filenames must match data.csv")
durations = []
for source in source_files:
    with wave.open(str(source), "rb") as original, wave.open(str(output_dir / source.name), "rb") as converted:
        if (original.getframerate(), original.getnchannels(), original.getsampwidth()) != (44_100, 1, 2):
            raise ValueError(f"Unexpected original WAV format: {source}")
        if (converted.getframerate(), converted.getnchannels(), converted.getsampwidth()) != (16_000, 1, 2):
            raise ValueError(f"Unexpected 16 kHz WAV format: {source.name}")
        duration = converted.getnframes() / converted.getframerate()
        if abs(original.getnframes() / original.getframerate() - duration) >= 1 / 16_000:
            raise ValueError(f"Duration changed: {source.name}")
        durations.append(duration)
frame["duration"] = durations
# Preserve existing measurements when refreshing the source manifest.
existing = hf_dir / "data.csv"
if existing.is_file():
    previous = pd.read_csv(existing).set_index("filename")
    for column in ("snr_original", "snr_16k"):
        if column in previous:
            frame[column] = frame["filename"].map(previous[column])
manifest_path = hf_dir / "data.csv"
partial_manifest = manifest_path.with_suffix(".csv.partial")
frame.to_csv(partial_manifest, index=False) 
partial_manifest.replace(manifest_path)
len(frame)